# 06 — Customer Segmentation and Business Insights

This notebook segments Instacart customers using shopping behavior, reorder tendency, basket size, timing patterns, and department affinity. The goal is to convert user behavior into actionable business strategies for personalization, retention, and basket growth.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [0]:
base_path = "/Volumes/workspace/default/instacart"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

prior = spark.read.parquet(f"{silver_path}/prior_order_details")
products = spark.read.parquet(f"{silver_path}/product_details")

### Why customer segmentation?

The reorder prediction model identifies which products a user is likely to buy again, while FP-Growth identifies basket-level cross-sell rules. Customer segmentation adds a business layer by grouping users with similar shopping patterns, allowing recommendation and retention strategies to differ by customer type.

1. Create order-level table

In [0]:
orders_only = (
    prior
    .select(
        "user_id",
        "order_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    )
    .dropDuplicates(["order_id"])
)

In [0]:
basket_size = (
    prior.groupBy("user_id", "order_id")
    .agg(F.count("*").alias("basket_size"))
)

2. Build user-level behavioral features

In [0]:
user_base = (
    prior.groupBy("user_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.count("*").alias("total_items"),
        F.countDistinct("product_id").alias("unique_products"),
        F.avg("reordered").alias("reorder_rate"),
        F.avg("add_to_cart_order").alias("avg_cart_position")
    )
    .withColumn("avg_basket_size", F.col("total_items") / F.col("total_orders"))
)

In [0]:
user_order_features = (
    orders_only.groupBy("user_id")
    .agg(
        F.avg("days_since_prior_order").alias("avg_days_between_orders"),
        F.stddev("days_since_prior_order").alias("std_days_between_orders"),
        F.avg("order_hour_of_day").alias("avg_order_hour"),
        F.avg(F.when(F.col("order_dow").isin([0, 6]), 1).otherwise(0)).alias("weekend_order_share"),
        F.avg(F.when((F.col("order_hour_of_day") >= 18) | (F.col("order_hour_of_day") <= 6), 1).otherwise(0)).alias("evening_night_order_share")
    )
)

In [0]:
user_basket_features = (
    basket_size.groupBy("user_id")
    .agg(
        F.avg("basket_size").alias("avg_basket_size_check"),
        F.stddev("basket_size").alias("std_basket_size"),
        F.max("basket_size").alias("max_basket_size")
    )
)

3. Add department affinity features

In [0]:
dept_counts = (
    prior.groupBy("user_id", "department")
    .agg(F.count("*").alias("dept_items"))
)

user_total_items = (
    prior.groupBy("user_id")
    .agg(F.count("*").alias("user_items_total"))
)

dept_share = (
    dept_counts
    .join(user_total_items, on="user_id", how="left")
    .withColumn("dept_share", F.col("dept_items") / F.col("user_items_total"))
)

In [0]:
dept_features = (
    dept_share.groupBy("user_id")
    .agg(
        F.max(F.when(F.col("department") == "produce", F.col("dept_share")).otherwise(0)).alias("produce_share"),
        F.max(F.when(F.col("department") == "dairy eggs", F.col("dept_share")).otherwise(0)).alias("dairy_share"),
        F.max(F.when(F.col("department") == "snacks", F.col("dept_share")).otherwise(0)).alias("snacks_share"),
        F.max(F.when(F.col("department") == "beverages", F.col("dept_share")).otherwise(0)).alias("beverages_share"),
        F.max(F.when(F.col("department") == "frozen", F.col("dept_share")).otherwise(0)).alias("frozen_share"),
        F.max(F.when(F.col("department") == "pantry", F.col("dept_share")).otherwise(0)).alias("pantry_share"),
        F.max(F.when(F.col("department") == "bakery", F.col("dept_share")).otherwise(0)).alias("bakery_share")
    )
)

In [0]:
w = Window.partitionBy("user_id").orderBy(F.desc("dept_items"))

user_top_department = (
    dept_counts
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "user_id",
        F.col("department").alias("top_department")
    )
)

4. Assemble segmentation table

In [0]:
user_segments_base = (
    user_base
    .join(user_order_features, on="user_id", how="left")
    .join(user_basket_features, on="user_id", how="left")
    .join(dept_features, on="user_id", how="left")
    .join(user_top_department, on="user_id", how="left")
)

In [0]:
fill_values = {
    "std_days_between_orders": 0.0,
    "std_basket_size": 0.0,
    "avg_days_between_orders": 0.0,
    "produce_share": 0.0,
    "dairy_share": 0.0,
    "snacks_share": 0.0,
    "beverages_share": 0.0,
    "frozen_share": 0.0,
    "pantry_share": 0.0,
    "bakery_share": 0.0
}

user_segments_base = user_segments_base.fillna(fill_values)

In [0]:
user_segments_base.write.mode("overwrite").parquet(f"{gold_path}/user_segmentation_features")

5. Select clustering features

In [0]:
cluster_features = [
    "total_orders",
    "total_items",
    "unique_products",
    "reorder_rate",
    "avg_basket_size",
    "avg_days_between_orders",
    "std_days_between_orders",
    "std_basket_size",
    "weekend_order_share",
    "evening_night_order_share",
    "produce_share",
    "dairy_share",
    "snacks_share",
    "beverages_share",
    "frozen_share",
    "pantry_share",
    "bakery_share"
]

6. Train K-Means for several K values

In [0]:
assembler = VectorAssembler(
    inputCols=cluster_features,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

### Practical clustering setup

K-Means is trained on a representative user sample due to Databricks Free Edition/serverless model-size limits. The fitted model is then applied to the full user base to assign segments. This keeps the workflow practical while preserving the ability to profile all users.

In [0]:
# Sample users for clustering evaluation to avoid serverless model-size limits
user_segments_sample = (
    user_segments_base
    .sample(withReplacement=False, fraction=0.10, seed=42)
)

print("Sample users:", user_segments_sample.count())

In [0]:
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette"
)

k_results = []

for k in [3, 4, 5, 6]:
    kmeans = KMeans(
        featuresCol="features",
        predictionCol="prediction",
        k=k,
        seed=42,
        maxIter=20
    )
    
    pipeline = Pipeline(stages=[assembler, scaler, kmeans])
    model = pipeline.fit(user_segments_sample)
    preds = model.transform(user_segments_sample)
    silhouette = evaluator.evaluate(preds)
    
    k_results.append({
        "k": k,
        "silhouette": silhouette
    })

k_results_df = spark.createDataFrame(k_results)
display(k_results_df)

Although silhouette score helps compare different values of K, the final number of segments should also be business-interpretable. A five-segment solution was selected because it balances separation quality with explainability.

7. Train final K-Means model

In [0]:
final_k = 5

kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=final_k,
    seed=42,
    maxIter=20
)

segmentation_pipeline = Pipeline(stages=[assembler, scaler, kmeans])
segmentation_model = segmentation_pipeline.fit(user_segments_sample)

In [0]:
segmented_users = segmentation_model.transform(user_segments_base)

display(segmented_users.select("user_id", "cluster").limit(20))

In [0]:
segmented_users.write.mode("overwrite").parquet(f"{gold_path}/segmented_users")
segmentation_model.write().overwrite().save(f"{gold_path}/customer_segmentation_kmeans_model")

8. Profile each segment

In [0]:
segment_profile = (
    segmented_users.groupBy("cluster")
    .agg(
        F.count("*").alias("num_users"),
        F.avg("total_orders").alias("avg_total_orders"),
        F.avg("total_items").alias("avg_total_items"),
        F.avg("unique_products").alias("avg_unique_products"),
        F.avg("reorder_rate").alias("avg_reorder_rate"),
        F.avg("avg_basket_size").alias("avg_basket_size"),
        F.avg("avg_days_between_orders").alias("avg_days_between_orders"),
        F.avg("weekend_order_share").alias("avg_weekend_order_share"),
        F.avg("evening_night_order_share").alias("avg_evening_night_order_share"),
        F.avg("produce_share").alias("avg_produce_share"),
        F.avg("dairy_share").alias("avg_dairy_share"),
        F.avg("snacks_share").alias("avg_snacks_share"),
        F.avg("beverages_share").alias("avg_beverages_share"),
        F.avg("frozen_share").alias("avg_frozen_share"),
        F.avg("pantry_share").alias("avg_pantry_share"),
        F.avg("bakery_share").alias("avg_bakery_share")
    )
    .orderBy("cluster")
)

display(segment_profile)

In [0]:
segment_profile.write.mode("overwrite").parquet(f"{gold_path}/segment_profile")

9. Find top departments by segment

In [0]:
user_cluster = segmented_users.select("user_id", "cluster")

segment_dept = (
    prior
    .join(user_cluster, on="user_id", how="inner")
    .groupBy("cluster", "department")
    .agg(F.count("*").alias("items"))
)

segment_total = (
    segment_dept.groupBy("cluster")
    .agg(F.sum("items").alias("cluster_items"))
)

segment_dept_share = (
    segment_dept
    .join(segment_total, on="cluster", how="left")
    .withColumn("department_share", F.col("items") / F.col("cluster_items"))
)

w_seg = Window.partitionBy("cluster").orderBy(F.desc("department_share"))

segment_top_departments = (
    segment_dept_share
    .withColumn("rank", F.row_number().over(w_seg))
    .filter(F.col("rank") <= 5)
    .orderBy("cluster", "rank")
)

display(segment_top_departments)

In [0]:
segment_top_departments.write.mode("overwrite").parquet(f"{gold_path}/segment_top_departments")

In [0]:
segment_name_map = {
    0: "Low-Frequency Small Basket Shoppers",
    1: "Snack & Beverage Routine Buyers",
    2: "Fresh Produce Loyalists",
    3: "Bulk Variety Shoppers",
    4: "High-Engagement Routine Reorderers"
}

segment_names_df = spark.createDataFrame(
    [(int(k), v) for k, v in segment_name_map.items()],
    ["cluster", "segment_name"]
)

segmented_users_named = (
    segmented_users
    .join(segment_names_df, on="cluster", how="left")
)

display(segmented_users_named.select("user_id", "cluster", "segment_name").limit(20))

segmented_users_named.write.mode("overwrite").parquet(
    f"{gold_path}/segmented_users_named"
)

In [0]:
business_strategy = spark.createDataFrame([
    (
        "Low-Frequency Small Basket Shoppers",
        "Low order frequency, smaller baskets, long reorder gaps, and the lowest reorder rate",
        "Use discovery recommendations, first-repeat incentives, personalized onboarding offers, and low-friction reorder prompts",
        "Activation and second-order conversion"
    ),
    (
        "Snack & Beverage Routine Buyers",
        "Small baskets with strong beverage and snack concentration and moderate repeat behavior",
        "Promote snack-drink bundles, party packs, checkout add-ons, and personalized impulse recommendations",
        "Attach rate and basket expansion"
    ),
    (
        "Fresh Produce Loyalists",
        "High produce share, moderate basket size, and strong fresh-food purchasing behavior",
        "Recommend fresh produce bundles, recipe-based baskets, dairy/produce pairings, and quality-focused messaging",
        "Fresh-category loyalty and basket size"
    ),
    (
        "Bulk Variety Shoppers",
        "Large baskets, high product variety, and broad department mix across produce, dairy, snacks, frozen, and beverages",
        "Use stock-up reminders, multi-category bundles, delivery threshold offers, and weekly planning recommendations",
        "Average order value"
    ),
    (
        "High-Engagement Routine Reorderers",
        "Highest order frequency, highest reorder rate, short reorder cycle, and strong repeat purchase behavior",
        "Offer one-click reorder baskets, subscription-style reminders, replenishment nudges, and personalized recurring lists",
        "Retention and reorder frequency"
    )
], ["segment_name", "behavior_summary", "recommended_strategy", "primary_business_metric"])

display(business_strategy)

business_strategy.write.mode("overwrite").parquet(
    f"{gold_path}/segment_business_strategy"
)

In [0]:
segment_profile_named = (
    segment_profile
    .join(segment_names_df, on="cluster", how="left")
    .join(business_strategy, on="segment_name", how="left")
    .select(
        "cluster",
        "segment_name",
        "num_users",
        "avg_total_orders",
        "avg_reorder_rate",
        "avg_basket_size",
        "avg_days_between_orders",
        "avg_produce_share",
        "avg_dairy_share",
        "avg_snacks_share",
        "avg_beverages_share",
        "behavior_summary",
        "recommended_strategy",
        "primary_business_metric"
    )
)

display(segment_profile_named)

segment_profile_named.write.mode("overwrite").parquet(
    f"{gold_path}/segment_profile_named"
)

## Business Insights Summary

Customer segmentation provides a strategy layer on top of the reorder prediction and FP-Growth recommendation components.

- Routine customers can be targeted with one-click reorder and replenishment reminders.
- Fresh-food-heavy customers can receive recipe-based bundles and produce-focused recommendations.
- Bulk shoppers can be targeted with pantry refill reminders and basket-size incentives.
- Snack and beverage shoppers can receive cross-sell and impulse add-on recommendations.
- Low-frequency users may benefit from discovery recommendations and repeat-purchase incentives.

Together, the three modeling components support a complete personalization workflow:

1. Reorder prediction: what the user is likely to buy again.
2. FP-Growth: what products are commonly bought together.
3. Segmentation: how personalization strategy should differ by customer type.